# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

## 1. Preprocessing

In [2]:
df_ns = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_scaled = pd.read_csv('../data/dayofweek.csv')
df_ns['dayofweek'] = df_scaled['dayofweek'].values

X = df_ns.drop('dayofweek', axis=1).values
y = df_ns['dayofweek'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (1348, 43) Test: (338, 43)


In [3]:
def evaluate_metrics(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
    
    print(f'accuracy is {acc:.5f}')
    print(f'precision is {prec:.5f}')
    print(f'recall is {rec:.5f}')
    print(f'roc_auc is {auc:.5f}')
    return model, y_pred

## 2. SVM

In [4]:
# Best params from Ex01 gridsearch
svm = SVC(kernel='rbf', C=10, gamma='auto', class_weight=None,
          probability=True, random_state=21)
svm_model, svm_pred = evaluate_metrics(svm, X_train, y_train, X_test, y_test)

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.98168


## 3. Decision tree

In [5]:
tree = DecisionTreeClassifier(max_depth=21, class_weight='balanced',
                               criterion='gini', random_state=21)
tree_model, tree_pred = evaluate_metrics(tree, X_train, y_train, X_test, y_test)

accuracy is 0.89053
precision is 0.89402
recall is 0.89053
roc_auc is 0.93977


## 4. Random forest

In [6]:
rf = RandomForestClassifier(n_estimators=50, max_depth=22, class_weight=None,
                             criterion='entropy', random_state=21)
rf_model, rf_pred = evaluate_metrics(rf, X_train, y_train, X_test, y_test)

accuracy is 0.93195
precision is 0.93556
recall is 0.93195
roc_auc is 0.99033


## 5. Predictions

In [7]:
# Best model is Random Forest
best_model = rf_model
y_pred = rf_pred

# Load original data with uid and labname for error analysis
raw = pd.read_csv('../data/checker_submits.csv', parse_dates=['timestamp'])
raw['dayofweek'] = raw['timestamp'].dt.dayofweek
raw['hour'] = raw['timestamp'].dt.hour

In [8]:
# Error analysis per weekday
errors = pd.DataFrame({'true': y_test, 'pred': y_pred})
errors['wrong'] = errors['true'] != errors['pred']

print('Error rate per weekday (0=Mon, 6=Sun):')
print(errors.groupby('true')['wrong'].mean().sort_values(ascending=False))

print('\nWorst weekday:', errors.groupby('true')['wrong'].mean().idxmax())

Error rate per weekday (0=Mon, 6=Sun):
true
0    0.222222
4    0.142857
1    0.090909
2    0.066667
5    0.055556
6    0.028169
3    0.025000
Name: wrong, dtype: float64

Worst weekday: 0


In [9]:
# Error analysis per labname and uid using full dataset
full_pred = best_model.predict(X)
raw_copy = df_ns.copy()
raw_copy['pred'] = full_pred
raw_copy['wrong'] = raw_copy['dayofweek'] != raw_copy['pred']

# Get labname and uid columns from checker_submits
submits = pd.read_csv('../data/checker_submits.csv', parse_dates=['timestamp'])
submits['dayofweek'] = submits['timestamp'].dt.dayofweek
submits['wrong'] = full_pred != submits['dayofweek'].values

print('Error rate per labname:')
print(submits.groupby('labname')['wrong'].mean().sort_values(ascending=False).head())

print('\nError rate per uid:')
print(submits.groupby('uid')['wrong'].mean().sort_values(ascending=False).head())

Error rate per labname:
labname
lab03      1.000000
laba06     0.041667
laba04     0.033708
laba06s    0.032787
lab05s     0.027778
Name: wrong, dtype: float64

Error rate per uid:
uid
user_23    0.250000
user_6     0.250000
user_22    0.142857
user_17    0.058824
user_27    0.043478
Name: wrong, dtype: float64


In [10]:
# Save model
with open('../data/metrics_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('Model saved!')

Model saved!


## 6. Function

In [11]:
def evaluate_models(models, params_list, X_train, y_train, X_test, y_test):
    results = {}
    for model_class, params in zip(models, params_list):
        model = model_class(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        name = model_class.__name__
        results[name] = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted'),
            'roc_auc': roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        }
    return results

# Test the function
models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
params_list = [
    {'kernel': 'rbf', 'C': 10, 'gamma': 'auto', 'probability': True, 'random_state': 21},
    {'max_depth': 21, 'criterion': 'gini', 'random_state': 21},
    {'n_estimators': 50, 'max_depth': 22, 'criterion': 'entropy', 'random_state': 21}
]

results = evaluate_models(models, params_list, X_train, y_train, X_test, y_test)
for name, metrics in results.items():
    print(f'\n{name}:')
    for metric, value in metrics.items():
        print(f'  {metric}: {value:.5f}')


SVC:
  accuracy: 0.88757
  precision: 0.89267
  recall: 0.88757
  roc_auc: 0.98168

DecisionTreeClassifier:
  accuracy: 0.86391
  precision: 0.86729
  recall: 0.86391
  roc_auc: 0.92297

RandomForestClassifier:
  accuracy: 0.93195
  precision: 0.93556
  recall: 0.93195
  roc_auc: 0.99033
